# บทที่ 4 สถิติเชิงอนุมานและการวิเคราะห์ข้อมูลเชิงสำรวจ
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch04_statistics/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch04_statistics/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch04_statistics/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch04_statistics'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'matplotlib': 'matplotlib', 'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'seaborn': 'seaborn>=0.12', 'statsmodels': 'statsmodels>=0.14'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_make_data.py`

Ch.4  Build the data files used in this chapter (all SYNTHETIC, SEED = 2026).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_make_data.py  จากโฟลเดอร์ของบท
__file__ = '01_make_data.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.4  Build the data files used in this chapter (all SYNTHETIC, SEED = 2026).

  data/raw/co60_counts.csv            1,000 one-second counts, Poisson with mean 5
                                      (exactly the recipe of exercise E4.2)
  data/raw/halpha_spectrum.csv        an H-alpha line: Gaussian, sigma = 0.020 nm
                                      (Doppler width at T = 1e4 K) + baseline + noise
  data/processed/galaxy10_features.csv
                                      the Chapter 3 galaxy table (stars removed, g_mag
                                      numeric) plus a light 'concentration' feature C
                                      (tab:invariant), drawn per class: smooth galaxies
                                      C ~ 4, spirals C ~ 2.8
Chapter 3 must have been run once (its 01_make_raw_data.py); if not, it is run here.
"""
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.constants as const

SEED = 2026
HERE = Path(__file__).resolve().parent
os.chdir(HERE)
rng = np.random.default_rng(SEED)
Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)

# --- Co-60 counts: exactly the recipe of E4.2 ---------------------------------
counts = rng.poisson(lam=5, size=1000)
pd.DataFrame({'interval': np.arange(1000), 'counts': counts}).to_csv(
    'data/raw/co60_counts.csv', index=False)

# --- H-alpha line with Doppler width at 1e4 K ---------------------------------
lam0 = 656.28                                    # nm
m_H = const.m_p + const.m_e
sigma_true = lam0 * np.sqrt(const.k * 1e4 / (m_H * const.c**2))
wavelength = np.linspace(lam0 - 0.25, lam0 + 0.25, 201)
intensity = (100 * np.exp(-(wavelength - lam0)**2 / (2 * sigma_true**2)) + 10
             + rng.normal(0, 2.0, wavelength.size))
pd.DataFrame({'wavelength_nm': np.round(wavelength, 5),
              'intensity': np.round(intensity, 3)}).to_csv('data/raw/halpha_spectrum.csv',
                                                           index=False)

# --- Galaxy10-style features ----------------------------------------------------
ch3 = HERE.parent / 'ch03_cleaning'
meta = ch3 / 'data' / 'raw' / 'galaxy10_metadata.csv'
if not meta.exists():
    subprocess.run([sys.executable, str(ch3 / '01_make_raw_data.py')], check=True)
df = pd.read_csv(meta)
df['g_mag'] = pd.to_numeric(df['g_mag'], errors='coerce')
df = df[df['r_mag'] >= 8].copy()                 # the five foreground stars (Ch.3)
C_mean = {'Round Smooth': 4.1, 'In-between Round Smooth': 3.8, 'Cigar Shaped Smooth': 3.6,
          'Edge-on with Bulge': 3.3, 'Edge-on without Bulge': 2.8, 'Barred Spiral': 3.0,
          'Unbarred Tight Spiral': 2.9, 'Unbarred Loose Spiral': 2.7,
          'Disturbed': 2.8, 'Merging': 2.9}
df['concentration'] = np.round(df['galaxy_class'].map(C_mean) + rng.normal(0, 0.3, len(df)), 3)
df.to_csv('data/processed/galaxy10_features.csv', index=False)

print(f"co60_counts.csv        : {counts.size} counts, mean {counts.mean():.3f}")
print(f"halpha_spectrum.csv    : sigma used {sigma_true:.4f} nm (T = 1e4 K)")
print(f"galaxy10_features.csv  : {len(df)} galaxies, columns {list(df.columns)}")

## `02_eda.py` — โค้ดที่ 4.1

Ch.4  Exploratory data analysis in five steps on the (SYNTHETIC) Galaxy10-style features.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_eda.py  จากโฟลเดอร์ของบท
__file__ = '02_eda.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.4  Exploratory data analysis in five steps on the (SYNTHETIC) Galaxy10-style features.

Book references: ssec:eda-workflow, code:eda
The listing is reproduced line for line; figures are saved instead of shown.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
os.chdir(Path(__file__).resolve().parent)

### โค้ดที่ 4.1

In [ ]:
# --- code:eda ---------------------------------------------------------------
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('data/processed/galaxy10_features.csv')

# 1: structure and summary numbers
df.info(); print(df.describe())

# 2: one variable at a time
sns.set_theme(style='whitegrid')
df.hist(bins=30, figsize=(10, 6))
plt.savefig('outputs/ch04_eda_1_hist.png', dpi=120)

# 3: outliers
plt.figure(); sns.boxplot(data=df[['r_mag', 'g_mag']])
plt.savefig('outputs/ch04_eda_2_box.png', dpi=120)

# 4: pairs of variables
sns.pairplot(df[['redshift', 'r_mag', 'g_mag', 'concentration']],
             diag_kind='hist', plot_kws={'alpha': 0.5, 's': 10})
plt.savefig('outputs/ch04_eda_3_pairs.png', dpi=100)

# 5: correlation heatmap
plt.figure()
corr = df.select_dtypes('number').corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.tight_layout(); plt.savefig('outputs/ch04_eda_4_corr.png', dpi=120)

print("\ncorrelation matrix:\n", corr.round(3))
print("saved outputs/ch04_eda_1_hist.png ... ch04_eda_4_corr.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `03_poisson_fit.py` — โค้ดที่ 4.2

Ch.4  Are Co-60 counts Poisson? variance/mean, histogram with the pmf, chi-squared test.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_poisson_fit.py  จากโฟลเดอร์ของบท
__file__ = '03_poisson_fit.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.4  Are Co-60 counts Poisson? variance/mean, histogram with the pmf, chi-squared test.

Book references: ssec:poisson-fit, ex:poisson-fit, code:poisson-fit
Counts are SYNTHETIC: rng.poisson(lam=5, size=1000), the recipe given in E4.2.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
os.chdir(Path(__file__).resolve().parent)

### โค้ดที่ 4.2

In [ ]:
# --- code:poisson-fit -------------------------------------------------------
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

counts = pd.read_csv('data/raw/co60_counts.csv')['counts'].values
lam_hat = counts.mean()
print(f"variance/mean = {counts.var(ddof=1)/lam_hat:.3f}")   # near 1?

# histogram with the Poisson pmf on top
n_max = counts.max() + 1
bins = np.arange(-0.5, n_max + 0.5, 1)
fig, ax = plt.subplots()
ax.hist(counts, bins=bins, density=True, alpha=0.6, label='data')
n_range = np.arange(0, n_max)
ax.plot(n_range, stats.poisson.pmf(n_range, mu=lam_hat), 'ro-', label='Poisson')
ax.set_xlabel('counts per 1 s'); ax.set_ylabel('probability'); ax.legend()

# chi-squared goodness of fit, keep bins with expected count >= 5
observed, _ = np.histogram(counts, bins=bins)
expected = len(counts) * stats.poisson.pmf(n_range, mu=lam_hat)
mask = expected >= 5
obs, exp = observed[mask], expected[mask]
exp = exp * obs.sum() / exp.sum()          # same total in both
chi2_stat, p = stats.chisquare(obs, f_exp=exp, ddof=1)   # 1 fitted parameter
print(f"chi^2 = {chi2_stat:.2f}, dof = {mask.sum() - 2}, p = {p:.3f}")

fig.savefig('outputs/ch04_poisson_fit.png', dpi=150)

# --- extra checks ---------------------------------------------------------------
print(f"\nmean = {lam_hat:.3f} counts/s from {counts.size} intervals; "
      f"kept {mask.sum()} of {n_range.size} bins (expected >= 5)")
# how much can variance/mean fluctuate for a true Poisson sample of this size?
rng = np.random.default_rng(1)
ratios = [ (x := rng.poisson(5, 1000)).var(ddof=1) / x.mean() for _ in range(2000)]
lo, hi = np.percentile(ratios, [2.5, 97.5])
print(f"for true Poisson data (N = 1000) variance/mean falls in [{lo:.3f}, {hi:.3f}] 95 % of the time")
print("saved outputs/ch04_poisson_fit.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `04_gaussian_fit.py` — โค้ดที่ 4.3

Ch.4  Fit a Gaussian + baseline to an H-alpha line and turn its width into a temperature.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_gaussian_fit.py  จากโฟลเดอร์ของบท
__file__ = '04_gaussian_fit.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.4  Fit a Gaussian + baseline to an H-alpha line and turn its width into a temperature.

Book references: ssec:gaussian-fit, ex:gaussian-fit, code:gaussian-fit
Text: at T = 1e4 K, sigma_lambda ~ 0.020 nm and FWHM = 2.355 sigma ~ 0.047 nm.
The spectrum is SYNTHETIC (made with sigma for T = 1e4 K, see 01_make_data.py).
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.constants as const

os.chdir(Path(__file__).resolve().parent)
spec = pd.read_csv('data/raw/halpha_spectrum.csv')
wavelength, intensity = spec['wavelength_nm'].values, spec['intensity'].values

# --- the estimate in the text -------------------------------------------------
lam0, T = 656.28, 1e4
m_H = const.m_p + const.m_e
sig = lam0 * np.sqrt(const.k * T / (m_H * const.c**2))
print(f"expected at T = 1e4 K: sigma = {sig:.4f} nm, FWHM = {2.355 * sig:.4f} nm "
      f"(2 sqrt(2 ln 2) = {2 * np.sqrt(2 * np.log(2)):.4f})")

### โค้ดที่ 4.3

In [ ]:
# --- code:gaussian-fit ------------------------------------------------------
from scipy.optimize import curve_fit

def gaussian(x, A, mu, sigma, b):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + b

p0 = [intensity.max() - intensity.min(),
      wavelength[intensity.argmax()], 0.02, intensity.min()]
popt, pcov = curve_fit(gaussian, wavelength, intensity, p0=p0)
A, mu, sigma, b = popt
print(f"line centre  mu    = {mu:.4f} nm")
print(f"line width   sigma = {sigma:.4f} +/- {np.sqrt(pcov[2, 2]):.4f} nm")
print(f"FWHM = {2.355 * sigma:.4f} nm")

# --- invert: temperature from the fitted width -----------------------------------
T_fit = m_H * const.c**2 / const.k * (sigma / mu)**2
dT = 2 * T_fit * np.sqrt(pcov[2, 2]) / sigma
print(f"temperature from the width: T = {T_fit:.0f} +/- {dT:.0f} K (generated with 10000 K)")

## `05_hypothesis_tests.py` — โค้ดที่ 4.4

Ch.4  Hypothesis tests on the Co-60 counts, and the numbers quoted in sec:inference.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_hypothesis_tests.py  จากโฟลเดอร์ของบท
__file__ = '05_hypothesis_tests.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.4  Hypothesis tests on the Co-60 counts, and the numbers quoted in sec:inference.

Book references: ssec:hypothesis-testing, ssec:chi2-ttest, code:chi2-ttest,
                 look-elsewhere effect, review question 4
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats

os.chdir(Path(__file__).resolve().parent)
# the Poisson example (code:poisson-fit) provides counts, obs and exp
counts = pd.read_csv('data/raw/co60_counts.csv')['counts'].values
lam_hat = counts.mean()
bins = np.arange(-0.5, counts.max() + 1.5, 1)
n_range = np.arange(0, counts.max() + 1)
observed, _ = np.histogram(counts, bins=bins)
expected = len(counts) * stats.poisson.pmf(n_range, mu=lam_hat)
mask = expected >= 5
obs, exp = observed[mask], expected[mask]
exp = exp * obs.sum() / exp.sum()

### โค้ดที่ 4.4

In [ ]:
# --- code:chi2-ttest --------------------------------------------------------
# chi-squared goodness of fit (reuse obs, exp from the Poisson example)
chi2_stat, p = stats.chisquare(obs, f_exp=exp, ddof=1)
print("H0 (Poisson):", "reject" if p < 0.05 else "do not reject",
      f"(p = {p:.3f})")

# one-sample t-test against the expected rate 5.0 counts/s
t_stat, p_t = stats.ttest_1samp(counts, popmean=5.0)
print(f"t = {t_stat:.3f}, p = {p_t:.3f}")

# --- what the t-test result means here -----------------------------------------
print(f"\nThese counts were GENERATED from a Poisson law with mean exactly 5.0, yet the"
      f" t-test gives p = {p_t:.3f}.")
print("With alpha = 0.05, true-H0 data are rejected 5 % of the time (type I error);"
      " this sample is one of those. One test on one data set is not a discovery.")
rng = np.random.default_rng(7)
p_sim = np.array([stats.ttest_1samp(rng.poisson(5, 1000), 5.0).pvalue for _ in range(4000)])
print(f"check with 4,000 simulated experiments: fraction with p < 0.05 = {np.mean(p_sim < 0.05):.3f}")

# --- numbers quoted in the section ------------------------------------------------
print(f"\n5 sigma, one-sided: alpha = {stats.norm.sf(5):.2e}  (text: ~3e-7)")
print(f"20 independent tests at 0.05: P(at least one false positive) = {1 - 0.95**20:.2f}  (text: ~64 %)")
print(f"50 mass windows at local p < 0.01: P(at least one) = {1 - 0.99**50:.2f}; "
      f"Bonferroni threshold 0.05/50 = {0.05 / 50}")

## `06_confidence_intervals.py` — โค้ดที่ 4.5

Ch.4  Confidence intervals: Poisson mean, Wilson interval for a fraction, what '95 %'

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_confidence_intervals.py  จากโฟลเดอร์ของบท
__file__ = '06_confidence_intervals.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.4  Confidence intervals: Poisson mean, Wilson interval for a fraction, what '95 %'
means (coverage), and why small samples need Student's t.

Book references: ssec:ci, def:ci, code:wilson, ssec:ci-basics, tab:t-vs-z, review question 7
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats

os.chdir(Path(__file__).resolve().parent)
counts = pd.read_csv('data/raw/co60_counts.csv')['counts'].values
n_bar, N = counts.mean(), counts.size
se = np.sqrt(n_bar / N)
print(f"Poisson mean: {n_bar:.3f} +/- {se:.3f}, 95 % CI [{n_bar - 1.96 * se:.3f}, {n_bar + 1.96 * se:.3f}]")

### โค้ดที่ 4.5

In [ ]:
# --- code:wilson ------------------------------------------------------------
from statsmodels.stats.proportion import proportion_confint

n_pass, n_total = 3, 40       # events passing a selection cut
ci_lo, ci_hi = proportion_confint(n_pass, n_total, alpha=0.05, method='wilson')
print(f"fraction = {n_pass/n_total:.3f}, 95% CI = [{ci_lo:.3f}, {ci_hi:.3f}]")
f = n_pass / n_total
half = 1.96 * np.sqrt(f * (1 - f) / n_total)
print(f"Gaussian (Wald) interval f +/- 1.96 sqrt(f(1-f)/n): [{f - half:.3f}, {f + half:.3f}]"
      "  <- lower edge below 0, impossible for a fraction")

# --- def:ci : 95 % is a property of the METHOD (ring-toss analogy) ---------------
rng = np.random.default_rng(11)
true_p, hits_w, hits_n = 3 / 40, 0, 0
for _ in range(20000):
    k = rng.binomial(40, true_p)
    lo, hi = proportion_confint(k, 40, alpha=0.05, method='wilson'); hits_w += lo <= true_p <= hi
    lo, hi = proportion_confint(k, 40, alpha=0.05, method='normal'); hits_n += lo <= true_p <= hi
print(f"coverage over 20,000 repeated experiments (true fraction 0.075, n = 40): "
      f"Wilson {hits_w / 20000:.3f}, Wald {hits_n / 20000:.3f}")

# --- tab:t-vs-z ------------------------------------------------------------------
print("\n  N   t_0.025,N-1   wider than z = 1.96 by")
for n in (5, 10, 30, 100):
    t = stats.t.ppf(0.975, df=n - 1)
    print(f"{n:4d}   {t:8.3f}      {100 * (t / 1.96 - 1):5.1f} %")

# coverage of mean +/- 1.96 s/sqrt(N) versus mean +/- t s/sqrt(N) for N = 5
hits_z = hits_t = 0
for _ in range(20000):
    x = rng.normal(0, 1, 5)
    h = x.std(ddof=1) / np.sqrt(5)
    hits_z += abs(x.mean()) < 1.96 * h
    hits_t += abs(x.mean()) < stats.t.ppf(0.975, 4) * h
print(f"N = 5: '95 %' interval with 1.96 covers {hits_z / 20000:.3f}, with t covers {hits_t / 20000:.3f}")

# --- review question 7: pendulum, five measurements ---------------------------------
mean, s, n = 2.013, 0.021, 5
sem = s / np.sqrt(n)
t = stats.t.ppf(0.975, n - 1)
print(f"\npendulum: SEM = {sem:.4f} s; 95 % CI = {mean} +/- {t * sem:.3f} s "
      f"(with 1.96: +/- {1.96 * sem:.3f} s, {100 * (1 - 1.96 / t):.0f} % too narrow)")
print(f"halving a CI needs {(2)**2}x the data (SE ~ 1/sqrt(N))")

## `07_metropolis_ising.py` — โค้ดที่ 4.6

Ch.4  Metropolis Monte Carlo for the 2D Ising model (L = 16), temperature scan.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 07_metropolis_ising.py  จากโฟลเดอร์ของบท
__file__ = '07_metropolis_ising.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.4  Metropolis Monte Carlo for the 2D Ising model (L = 16), temperature scan.

Book references: ssec:metropolis, code:metropolis, tab:ising-T-scan, exercise E4.4
The listing is reproduced line for line (it takes about 2 minutes in pure Python).
Results are compared with Onsager's exact infinite-lattice energy and saved to
outputs/ch04_ising_metropolis_L16.csv for exercise E4.4 and script 08.
"""
import os
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.special import ellipk

os.chdir(Path(__file__).resolve().parent)
t0 = time.perf_counter()

### โค้ดที่ 4.6

In [ ]:
# --- code:metropolis --------------------------------------------------------
import numpy as np
rng = np.random.default_rng(seed=2026)

def init_lattice(L, hot_start=True):
    if hot_start:
        return rng.choice([-1, 1], size=(L, L))
    return np.ones((L, L), dtype=int)

def total_energy(lattice):
    # each bond counted once: one neighbour along each axis
    return -np.sum(lattice * (np.roll(lattice, 1, 0) + np.roll(lattice, 1, 1)))

def metropolis_step(lattice, beta):
    """One sweep = L*L attempted single-spin flips."""
    L = lattice.shape[0]
    for _ in range(L * L):
        i, j = rng.integers(0, L, size=2)
        s = lattice[i, j]
        nb = (lattice[(i+1) % L, j] + lattice[(i-1) % L, j]
            + lattice[i, (j+1) % L] + lattice[i, (j-1) % L])
        dE = 2 * s * nb
        if dE <= 0 or rng.random() < np.exp(-beta * dE):
            lattice[i, j] = -s
    return lattice

def simulate_T(L, T, n_thermalize, n_measure):
    lattice = init_lattice(L)
    beta = 1.0 / T
    for _ in range(n_thermalize):          # discard: not yet in equilibrium
        metropolis_step(lattice, beta)
    energies = []
    for _ in range(n_measure):
        metropolis_step(lattice, beta)
        energies.append(total_energy(lattice))
    energies = np.asarray(energies)
    E_mean = energies.mean() / (L * L)
    Cv = energies.var() / (T**2 * L * L)   # fluctuation formula
    return E_mean, Cv

L = 16
T_values = np.linspace(1.5, 3.5, 21)
E_means, Cvs = [], []
for T in T_values:
    Em, Cv = simulate_T(L, T, n_thermalize=1000, n_measure=2000)
    E_means.append(Em); Cvs.append(Cv)

# --- results --------------------------------------------------------------------
elapsed = time.perf_counter() - t0


def onsager_energy(T):
    """Exact energy per spin of the infinite 2D Ising model (J = k_B = 1)."""
    b = 1.0 / T
    k = 2 * np.sinh(2 * b) / np.cosh(2 * b)**2
    return -(1 / np.tanh(2 * b)) * (1 + 2 / np.pi * (2 * np.tanh(2 * b)**2 - 1) * ellipk(k**2))


Tc = 2 / np.log(1 + np.sqrt(2))
res = pd.DataFrame({'T': T_values, 'E_per_spin': E_means, 'Cv_per_spin': Cvs,
                    'E_onsager': [onsager_energy(T) for T in T_values]})
res.round(4).to_csv('outputs/ch04_ising_metropolis_L16.csv', index=False)
print(res.round(3).to_string(index=False))
print(f"\nrun time: {elapsed:.0f} s for 21 temperatures")
print(f"T_c (Onsager) = {Tc:.4f}; exact E/N just above T_c = {onsager_energy(Tc * (1 + 1e-9)):.4f}, -sqrt(2) = {-np.sqrt(2):.4f}")
i = int(np.argmax(Cvs))
print(f"C_v peak of this L = 16 run at T = {T_values[i]:.2f} (grid step 0.1), C_v/N = {Cvs[i]:.2f}")
for T in (1.5, 2.0, 3.5):
    j = int(np.argmin(abs(T_values - T)))
    print(f"E/N at T = {T}: {E_means[j]:.3f} (Onsager, infinite lattice: {onsager_energy(T):.3f})")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
Tf = np.linspace(1.5, 3.5, 200)
ax[0].plot(T_values, E_means, 'o', label='Metropolis, L = 16')
ax[0].plot(Tf, [onsager_energy(T) for T in Tf], '-', label='Onsager, L = infinity')
ax[0].set_xlabel('T'); ax[0].set_ylabel('<E>/N'); ax[0].legend()
ax[1].plot(T_values, Cvs, 'o-'); ax[1].axvline(Tc, ls=':', color='gray')
ax[1].set_xlabel('T'); ax[1].set_ylabel('C_v/N')
for a in ax: a.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('outputs/ch04_ising_metropolis.png', dpi=150)
print("saved outputs/ch04_ising_metropolis.png and outputs/ch04_ising_metropolis_L16.csv")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `08_wang_landau_ising.py` — โค้ดที่ 4.7

Ch.4  Wang-Landau density of states for the 2D Ising model (L = 4), with checks.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 08_wang_landau_ising.py  จากโฟลเดอร์ของบท
__file__ = '08_wang_landau_ising.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.4  Wang-Landau density of states for the 2D Ising model (L = 4), with checks.

Book references: ssec:wang-landau, def:wl, code:wl, tab:wl-thermo, exercise E4.5
The listing is reproduced line for line. Checks added:
  * sum of g(E) against 2^16 for seeds 1-8, at ln f = 1e-6 and 1e-8 (error saturation)
  * g(E) against the exact values from enumerating all 65,536 states
  * entropy per spin -> ln 2 at high T
"""
import itertools
import os
from pathlib import Path

import numpy as np

os.chdir(Path(__file__).resolve().parent)
rng = np.random.default_rng(seed=2026)

# from code:metropolis
def total_energy(lattice):
    # each bond counted once: one neighbour along each axis
    return -np.sum(lattice * (np.roll(lattice, 1, 0) + np.roll(lattice, 1, 1)))

### โค้ดที่ 4.7

In [ ]:
# --- code:wl ----------------------------------------------------------------
def wang_landau_2d_ising(L, ln_f_final=1e-8, flatness=0.8,
                         check_every=1000, max_sweeps=10**7):
    lattice = rng.choice([-1, 1], size=(L, L))
    N = L * L
    E_min = -2 * N
    E_bins = np.arange(E_min, 2 * N + 1, 4)
    log_g = np.zeros(len(E_bins))
    H = np.zeros(len(E_bins), dtype=int)
    visited = np.zeros(len(E_bins), dtype=bool)   # some bins never occur
    ln_f = 1.0
    idx = lambda E: (E - E_min) // 4

    E_curr = total_energy(lattice)
    sweep = 0
    while ln_f > ln_f_final and sweep < max_sweeps:
        for _ in range(N):
            i, j = rng.integers(0, L, size=2)
            s = lattice[i, j]
            nb = (lattice[(i+1) % L, j] + lattice[(i-1) % L, j]
                + lattice[i, (j+1) % L] + lattice[i, (j-1) % L])
            E_new = E_curr + 2 * s * nb
            if rng.random() < np.exp(log_g[idx(E_curr)] - log_g[idx(E_new)]):
                lattice[i, j] = -s
                E_curr = E_new
            k = idx(E_curr)
            log_g[k] += ln_f
            H[k] += 1
            visited[k] = True
        sweep += 1
        if sweep % check_every == 0:
            h = H[visited]
            if h.min() >= flatness * h.mean():    # histogram is flat enough
                ln_f /= 2.0
                H[:] = 0
    E_bins, log_g = E_bins[visited], log_g[visited]
    log_g += np.log(2.0) - log_g[0]               # anchor: g(E_min) = 2
    return E_bins, log_g, sweep

def thermodynamic(E_bins, log_g, T_values):
    res = {'T': T_values, 'E_mean': [], 'Cv': [], 'S': []}
    for T in T_values:
        beta = 1.0 / T
        log_w = log_g - beta * E_bins
        log_w_max = log_w.max()                   # log-sum-exp trick
        w = np.exp(log_w - log_w_max)
        Z = w.sum()
        E_mean = (E_bins * w).sum() / Z
        E2_mean = (E_bins**2 * w).sum() / Z
        res['E_mean'].append(E_mean)
        res['Cv'].append(beta**2 * (E2_mean - E_mean**2))
        res['S'].append(np.log(Z) + log_w_max + beta * E_mean)   # S/k_B
    return res

E_bins, log_g, n_sweeps = wang_landau_2d_ising(L=4, ln_f_final=1e-6)
print(np.exp(log_g).sum(), 2**16)               # sanity check

# --- exact density of states by enumeration ------------------------------------
states = np.array(list(itertools.product([-1, 1], repeat=16))).reshape(-1, 4, 4)
E_all = -np.sum(states * (np.roll(states, 1, 1) + np.roll(states, 1, 2)), axis=(1, 2))
vals, cnt = np.unique(E_all, return_counts=True)
exact = dict(zip(vals.tolist(), cnt.tolist()))
print("exact g(E):", exact)
print("no states at E = -28 or +28:", -28 not in exact and 28 not in exact)
print(f"this run (seed 2026): sum g = {np.exp(log_g).sum():.0f} "
      f"({100 * (np.exp(log_g).sum() / 2**16 - 1):+.1f} %), {n_sweeps} sweeps")

# --- several seeds and two final ln f -------------------------------------------
print("\nseed  ln_f=1e-6: sum error  g(-24)   g(-20) | ln_f=1e-8: sum error")
for seed in range(1, 9):
    out = []
    for lnf in (1e-6, 1e-8):
        rng = np.random.default_rng(seed)
        Eb, lg, _ = wang_landau_2d_ising(L=4, ln_f_final=lnf)
        out.append((Eb, np.exp(lg)))
    (Eb, g6), (_, g8) = out
    d = dict(zip(Eb.tolist(), g6))
    print(f"{seed:4d}       {100 * (g6.sum() / 2**16 - 1):+6.1f} %  {d[-24]:6.1f}  {d[-20]:6.1f}"
          f"  |        {100 * (g8.sum() / 2**16 - 1):+6.1f} %")
print("-> lowering ln f from 1e-6 to 1e-8 changes almost nothing: error saturation")

# where does the error sit? relative error of g(E) along the energy axis (seed 3)
rng = np.random.default_rng(3)
Eb, lg, _ = wang_landau_2d_ising(L=4, ln_f_final=1e-6)
print("\nseed 3, error of g(E) vs exact, from the anchor E_min outward:")
print("  " + "  ".join(f"{e}:{100 * (np.exp(l) / exact[e] - 1):+.0f}%" for e, l in zip(Eb, lg)))
print("-> small next to the anchor g(-32) = 2, larger far from it; g(E) and g(-E) can differ"
      " by tens of percent without any bug")

# --- thermodynamics from g(E) of the seed-2026 run (E_bins, log_g of code:wl) ------
T_values = np.linspace(0.5, 5.0, 91)
res = thermodynamic(E_bins, log_g, T_values)
N = 16
k = int(np.argmax(res['Cv']))
print(f"\nC_v/N peak at T = {T_values[k]:.2f} (L = 4); S/N at T = 5: {res['S'][-1] / N:.3f}, "
      f"ln 2 = {np.log(2):.3f}; S/N at T = 50: {thermodynamic(E_bins, log_g, [50.0])['S'][0] / N:.3f}")
print("(at infinite T, S/N = ln(sum g)/N; it misses ln 2 by exactly the error of the sum)")

## `09_reporting.py`

Ch.4  Reporting a measurement: rounding rules, stat/syst split, error propagation.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 09_reporting.py  จากโฟลเดอร์ของบท
__file__ = '09_reporting.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.4  Reporting a measurement: rounding rules, stat/syst split, error propagation.

Book references: ssec:reporting, tab:reporting-examples, ssec:uq-PI, exercise E4.6
Rule used in the book (Taylor): round the uncertainty to one significant figure, or two
if its first digit is 1; then round the value to the same decimal place.
"""
import math

import numpy as np


def report(value, unc, unit=""):
    """Format value +/- uncertainty following the rule in ssec:reporting."""
    exp10 = math.floor(math.log10(abs(unc)))
    first = int(unc / 10**exp10)
    digits = 2 if first == 1 else 1
    dec = -(exp10 - digits + 1)                         # decimal places to keep
    u = round(unc, dec); v = round(value, dec)
    if abs(v) >= 1e4 or abs(v) < 1e-3:                  # scientific notation
        e = math.floor(math.log10(abs(v)))
        s = 10**e
        d = max(dec + e, 0)
        return f"({v / s:.{d}f} ± {u / s:.{d}f}) × 10^{e} {unit}".strip()
    d = max(dec, 0)
    return f"({v:.{d}f} ± {u:.{d}f}) {unit}".strip()


print("tab:reporting-examples, recomputed from the 'wrong' versions:")
for v, u, unit in [(656.28135, 0.00521, "nm"), (9.81254, 0.02, "m/s^2"),
                   (342156.7, 15234.2, "m/s")]:
    print(f"  {v} ± {u}  ->  {report(v, u, unit)}")
print(f"rounding 0.14 to 0.1 changes it by {100 * 0.04 / 0.14:.0f} % (text: almost 30 %)")

# --- Co-60 half-life (text and E4.6) -------------------------------------------------
T, stat, syst = 5.2713, 0.0042, 0.005
print(f"\nT1/2(Co-60) = {T:.3f} ± {round(stat, 3)} (stat) ± {syst} (syst) years")
ref_d, ref_ud = 1925.28, 0.14
ref, uref = ref_d / 365.25, ref_ud / 365.25
print(f"reference: {ref_d} ± {ref_ud} d = {ref:.4f} ± {uref:.4f} y (1 y = 365.25 d)")
for days in (365.2422, 365.0):
    shift = ref_d / days - ref
    print(f"  with 1 y = {days} d: {ref_d / days:.4f} y, shift {shift:+.4f} y "
          f"({'larger' if abs(shift) > uref else 'smaller'} than the reference uncertainty {uref:.4f} y)")
tot = math.hypot(stat, syst)
print(f"95 % CI (statistical only): {T} ± {1.96 * stat:.4f} -> [{T - 1.96 * stat:.3f}, {T + 1.96 * stat:.3f}] y")
print(f"difference measured - reference = {T - ref:+.4f} y = {(T - ref) / stat:+.2f} sigma_stat")
print(f"combined uncertainty sqrt(stat^2 + syst^2) = {tot:.4f} y; halving stat (4x more counting)"
      f" gives {math.hypot(stat / 2, syst):.4f} y: only {100 * (1 - math.hypot(stat / 2, syst) / tot):.0f} % better")

# --- error propagation: pendulum g = 4 pi^2 l / T^2 -------------------------------------
l, sl = 1.000, 0.001            # m
Tp, sT = 2.007, 0.02 * 2.007 / 2  # s, 1 % uncertainty
g = 4 * np.pi**2 * l / Tp**2
sg = g * np.hypot(sl / l, 2 * sT / Tp)
print(f"\npendulum: T known to 1 %, l to 0.1 % -> g = {report(g, sg, 'm/s^2')}; "
      f"relative error of g = {100 * sg / g:.2f} % (the period contributes 2 x 1 %)")
rng = np.random.default_rng(5)
gs = 4 * np.pi**2 * rng.normal(l, sl, 200000) / rng.normal(Tp, sT, 200000)**2
print(f"Monte Carlo check: std(g)/g = {100 * gs.std() / gs.mean():.2f} %")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E4_2_counts_eda.py`

E4.2  Explore the Co-60 counts: describe(), variance/mean, histogram + pmf, Q-Q plot

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E4_2_counts_eda.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E4_2_counts_eda.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E4.2  Explore the Co-60 counts: describe(), variance/mean, histogram + pmf, Q-Q plot
against the Poisson distribution, chi-squared test.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats

os.chdir(Path(__file__).resolve().parent.parent)
counts = pd.read_csv('data/raw/co60_counts.csv')['counts']
print(counts.describe().round(3).to_string())
lam = counts.mean()
print(f"variance/mean = {counts.var() / lam:.3f}")

# Q-Q plot: sorted data against Poisson quantiles at the same plotting positions
q = (np.arange(1, counts.size + 1) - 0.5) / counts.size
theo = stats.poisson.ppf(q, mu=lam)
jit = np.random.default_rng(0).uniform(-0.15, 0.15, counts.size)   # separate ties
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
bins = np.arange(-0.5, counts.max() + 1.5)
ax[0].hist(counts, bins=bins, density=True, alpha=0.6, label='data')
k = np.arange(counts.max() + 1)
ax[0].plot(k, stats.poisson.pmf(k, lam), 'ro-', label=f'Poisson, mean {lam:.2f}')
ax[0].set_xlabel('counts per 1 s'); ax[0].legend()
ax[1].plot(theo + jit, np.sort(counts) + jit, '.', ms=2)
ax[1].plot([0, counts.max()], [0, counts.max()], 'k--')
ax[1].set_xlabel('Poisson quantile'); ax[1].set_ylabel('data quantile'); ax[1].set_title('Q-Q plot')
fig.tight_layout(); fig.savefig('outputs/E4_2_counts.png', dpi=150)
frac_on_line = np.mean(np.sort(counts.values) == theo)
print(f"{100 * frac_on_line:.1f} % of the sorted counts equal the Poisson quantile exactly")
print("conclusion: the data do not contradict a Poisson law (not: 'are certainly Poisson')")
print("saved outputs/E4_2_counts.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E4_3_chi2_interpretation.py`

E4.3  What a small or a large p-value can mean physically: simulate counts with a

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E4_3_chi2_interpretation.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E4_3_chi2_interpretation.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E4.3  What a small or a large p-value can mean physically: simulate counts with a
dead-time counter (variance < mean) and with a drifting background (variance > mean),
and ask how far from Poisson the real data could be (CI of variance/mean).
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats

os.chdir(Path(__file__).resolve().parent.parent)
rng = np.random.default_rng(43)


def chi2_poisson(counts):
    lam = counts.mean()
    bins = np.arange(-0.5, counts.max() + 1.5)
    k = np.arange(counts.max() + 1)
    obs, _ = np.histogram(counts, bins)
    exp = len(counts) * stats.poisson.pmf(k, lam)
    m = exp >= 5
    o, e = obs[m], exp[m] * obs[m].sum() / exp[m].sum()
    return stats.chisquare(o, e, ddof=1).pvalue


def dead_time_counts(rate, tau, n, T=1.0):
    """Paralyzable-free (non-paralyzable) counter: events closer than tau are lost."""
    out = []
    for _ in range(n):
        t = np.sort(rng.uniform(0, T, rng.poisson(rate * T)))
        kept, last = 0, -np.inf
        for ti in t:
            if ti - last >= tau:
                kept += 1; last = ti
        out.append(kept)
    return np.array(out)


data = pd.read_csv('data/raw/co60_counts.csv')['counts'].values
cases = {
    'Co-60 data (this chapter)': data,
    'dead time: rate 12/s, tau 0.05 s': dead_time_counts(12, 0.05, 1000),
    'background drifting 3 -> 7 /s': rng.poisson(np.linspace(3, 7, 1000)),
}
print(f"{'case':36s} {'mean':>6s} {'var/mean':>9s} {'chi2 p':>8s}")
for name, c in cases.items():
    print(f"{name:36s} {c.mean():6.2f} {c.var(ddof=1) / c.mean():9.3f} {chi2_poisson(c):8.3g}")

# (c) a high p does not prove Poisson: 95 % CI of variance/mean by bootstrap
boot = []
for _ in range(4000):
    s = rng.choice(data, data.size)
    boot.append(s.var(ddof=1) / s.mean())
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"\nCo-60 data: variance/mean 95 % bootstrap CI [{lo:.3f}, {hi:.3f}] -> deviations from"
      " Poisson of up to ~8 % either way are not excluded")

## `exercises/E4_4_metropolis_thermalization.py` — โค้ดที่ 4.6

E4.4  Thermalisation at T = 2.0 (L = 16) from a random start, and the C_v peak for L = 8

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E4_4_metropolis_thermalization.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E4_4_metropolis_thermalization.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E4.4  Thermalisation at T = 2.0 (L = 16) from a random start, and the C_v peak for L = 8
compared with the L = 16 scan of 07_metropolis_ising.py.
Uses the functions of code:metropolis unchanged.
"""
import os
import re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
# take the function definitions from script 07 (code:metropolis), not the scan itself
src = (HERE / '07_metropolis_ising.py').read_text()
defs = src[src.index('import numpy as np\nrng'):src.index('L = 16\nT_values')]
exec(defs)                                   # defines rng, init_lattice, total_energy, ...

# --- energy trace at T = 2.0 from a hot start -----------------------------------------
L, T = 16, 2.0
lat = init_lattice(L)
trace = []
for sweep in range(1500):
    metropolis_step(lat, 1 / T)
    trace.append(total_energy(lat) / L**2)
trace = np.array(trace)
eq = trace[500:].mean()
first = int(np.argmax(np.abs(trace - eq) < 0.02))
print(f"T = 2.0, L = 16: E/N after 500 sweeps averages {eq:.3f}; first within 0.02 of it at"
      f" sweep {first}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(trace, lw=0.8); ax.axhline(eq, ls=':', color='k')
ax.set_xlabel('sweep'); ax.set_ylabel('E/N'); ax.set_title('thermalisation, T = 2.0, L = 16')
ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig('outputs/E4_4_thermalization.png', dpi=150)

# --- L = 8 scan with the same settings as the L = 16 scan ------------------------------
T_values = np.linspace(1.5, 3.5, 21)
res8 = [simulate_T(8, T, n_thermalize=1000, n_measure=2000) for T in T_values]
Cv8 = np.array([r[1] for r in res8])
pd.DataFrame({'T': T_values, 'E_per_spin': [r[0] for r in res8], 'Cv_per_spin': Cv8}).round(4) \
    .to_csv('outputs/ch04_ising_metropolis_L8.csv', index=False)   # used by E4_5b
res16 = pd.read_csv('outputs/ch04_ising_metropolis_L16.csv')
i8, i16 = int(np.argmax(Cv8)), int(res16['Cv_per_spin'].idxmax())
print(f"C_v/N peak: L = 8 at T = {T_values[i8]:.1f} (height {Cv8[i8]:.2f}); "
      f"L = 16 at T = {res16['T'][i16]:.1f} (height {res16['Cv_per_spin'][i16]:.2f})")
print("-> the smaller lattice gives a lower, broader peak; both lie near T_c = 2.269,"
      " limited by the 0.1 temperature grid")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E4_5_wang_landau_thermo.py`

E4.5  Thermodynamics of the 4 x 4 Ising model from Wang-Landau g(E), against the EXACT

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E4_5_wang_landau_thermo.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E4_5_wang_landau_thermo.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E4.5  Thermodynamics of the 4 x 4 Ising model from Wang-Landau g(E), against the EXACT
result from enumerating all 2^16 states.
"""
import itertools
import os
import re
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
src = (HERE / '08_wang_landau_ising.py').read_text()
exec(src[src.index('rng = np.random.default_rng(seed=2026)'):src.index('E_bins, log_g, n_sweeps = wang_landau_2d_ising')])

E_bins, log_g, n_sweeps = wang_landau_2d_ising(L=4, ln_f_final=1e-6)
states = np.array(list(itertools.product([-1, 1], repeat=16))).reshape(-1, 4, 4)
E_all = -np.sum(states * (np.roll(states, 1, 1) + np.roll(states, 1, 2)), axis=(1, 2))
Ex, gx = np.unique(E_all, return_counts=True)

T = np.array([0.5, 1.0, 1.5, 2.0, 2.27, 2.5, 3.0, 4.0, 5.0])
wl, ex = thermodynamic(E_bins, log_g, T), thermodynamic(Ex, np.log(gx), T)
N = 16
print("   T    <E>/N WL  exact     C_v/N WL  exact     S/N WL  exact")
for i, t in enumerate(T):
    print(f"{t:5.2f}  {wl['E_mean'][i] / N:8.4f} {ex['E_mean'][i] / N:7.4f}   "
          f"{wl['Cv'][i] / N:8.4f} {ex['Cv'][i] / N:7.4f}   {wl['S'][i] / N:7.4f} {ex['S'][i] / N:7.4f}")
Tf = np.linspace(0.5, 5, 451)
kw, ke = np.argmax(thermodynamic(E_bins, log_g, Tf)['Cv']), np.argmax(thermodynamic(Ex, np.log(gx), Tf)['Cv'])
print(f"C_v peak: WL T = {Tf[kw]:.2f}, exact T = {Tf[ke]:.2f} (L = 4, far from T_c = 2.269: finite size)")
print(f"S/N -> ln 2 = {np.log(2):.4f} as T -> infinity (exact S/N at T = 5: {ex['S'][-1] / N:.4f})")
rel = lambda q: np.max(np.abs(np.array(wl[q]) / np.array(ex[q]) - 1)[1:])
print(f"largest relative difference WL vs exact for T >= 1: <E> {100 * rel('E_mean'):.1f} %, "
      f"C_v {100 * rel('Cv'):.1f} %, S {100 * rel('S'):.1f} %")
print("8 x 8 lattice and comparison with the exact finite-lattice result: see E4_5b_wang_landau_8x8.py")

## `exercises/E4_5b_wang_landau_8x8.py` — โค้ดที่ 4.7

E4.5 (second part)  Wang-Landau for the 8 x 8 Ising model with ln f = 1e-8.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E4_5b_wang_landau_8x8.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E4_5b_wang_landau_8x8.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E4.5 (second part)  Wang-Landau for the 8 x 8 Ising model with ln f = 1e-8.

Checks three statements of the book on the 8 x 8 lattice:
  * how long code:wl takes in pure Python (compared with 4 x 4 at the same ln f),
  * whether <E>/N and C_v/N agree with the Metropolis L = 8 scan of E4.4,
  * how close the result is to the EXACT answer.

The exact reference is Kaufman's closed-form partition function of the finite
L x L lattice with periodic boundaries (B. Kaufman, Phys. Rev. 76, 1232 (1949);
A. E. Ferdinand and M. E. Fisher, Phys. Rev. 185, 832 (1969)).  The function
lnZ_kaufman() is checked below against full enumeration of the 4 x 4 lattice.
"""
import itertools
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
src = (HERE / '08_wang_landau_ising.py').read_text()
exec(src[src.index('rng = np.random.default_rng(seed=2026)'):src.index('E_bins, log_g, n_sweeps = wang_landau_2d_ising')])


def lnZ_kaufman(L, T):
    """Exact ln Z of the L x L periodic 2D Ising model (J = k_B = 1)."""
    K = 1.0 / T
    l = np.arange(2 * L)
    gam = np.arccosh(np.cosh(2 * K) / np.tanh(2 * K) - np.cos(np.pi * l / L))
    gam[0] = 2 * K + np.log(np.tanh(K))            # gamma_0 keeps its sign
    terms = []
    for g_set in (gam[1::2], gam[0::2]):
        for f in (np.cosh, np.sinh):
            v = 2 * f(L * g_set / 2)
            terms.append((np.sum(np.log(np.abs(v))), np.prod(np.sign(v))))
    top = max(t[0] for t in terms)
    s = sum(sg * np.exp(lg - top) for lg, sg in terms)
    return np.log(0.5) + (L * L / 2) * np.log(2 * np.sinh(2 * K)) + top + np.log(s)


def exact_E_Cv(L, T, h=1e-4):
    """<E>/N and C_v/N from derivatives of ln Z with respect to beta."""
    b = 1.0 / T
    f = lambda bb: lnZ_kaufman(L, 1.0 / bb)
    E = -(f(b + h) - f(b - h)) / (2 * h)
    C = b**2 * (f(b + h) - 2 * f(b) + f(b - h)) / h**2
    return E / (L * L), C / (L * L)


# --- 1. check the exact formula against enumeration of all 2^16 states of 4 x 4 --------
states = np.array(list(itertools.product([-1, 1], repeat=16))).reshape(-1, 4, 4)
E_all = -np.sum(states * (np.roll(states, 1, 1) + np.roll(states, 1, 2)), axis=(1, 2))
Ex, gx = np.unique(E_all, return_counts=True)
worst = max(abs(np.log(np.sum(gx * np.exp(-Ex / T))) - lnZ_kaufman(4, T)) for T in (1.0, 2.27, 5.0))
print(f"Kaufman ln Z vs enumeration (4 x 4): largest difference {worst:.1e}")

# --- 2. timing: 4 x 4 and 8 x 8 at ln f = 1e-8 -----------------------------------------
t0 = time.time()
E4, lg4, n4 = wang_landau_2d_ising(L=4, ln_f_final=1e-8)
t4 = time.time() - t0
t0 = time.time()
E8, lg8, n8 = wang_landau_2d_ising(L=8, ln_f_final=1e-8)
t8 = time.time() - t0
print(f"4 x 4: {n4:7d} sweeps = {16 * n4:10,d} spin-flip attempts, {t4:5.1f} s")
print(f"8 x 8: {n8:7d} sweeps = {64 * n8:10,d} spin-flip attempts, {t8:5.1f} s")
print(f"-> 4 times more spins, {64 * n8 / (16 * n4):.0f} times more attempts "
      f"({len(E8)} instead of {len(E4)} energy levels)")
sum_err = np.exp(np.logaddexp.reduce(lg8) - 64 * np.log(2)) - 1
print(f"8 x 8: sum of g(E) differs from 2^64 by {100 * sum_err:+.1f} %")

# --- 3. thermodynamics: Wang-Landau, Metropolis (E4.4), exact ---------------------------
T = np.array([1.5, 2.0, 2.3, 2.4, 2.5, 3.0, 3.5])
wl = thermodynamic(E8, lg8, T)
ex = np.array([exact_E_Cv(8, t) for t in T])
met_file = Path('outputs/ch04_ising_metropolis_L8.csv')
met = pd.read_csv(met_file).set_index('T') if met_file.exists() else None
print("\n   T   <E>/N: WL    Metro   exact   |  C_v/N: WL   Metro   exact")
for i, t in enumerate(T):
    mE = mC = float('nan')
    if met is not None:
        row = met.loc[np.isclose(met.index, t)]
        mE, mC = row['E_per_spin'].iloc[0], row['Cv_per_spin'].iloc[0]
    print(f"{t:5.2f}  {wl['E_mean'][i] / 64:8.4f} {mE:8.4f} {ex[i, 0]:7.4f}   |  "
          f"{wl['Cv'][i] / 64:7.4f} {mC:7.4f} {ex[i, 1]:7.4f}")
if met is None:
    print("(run E4_4_metropolis_thermalization.py first to fill the Metropolis column)")

Tf = np.linspace(1.8, 3.0, 1201)
k_wl = np.argmax(thermodynamic(E8, lg8, Tf)['Cv'])
C_ex = np.array([exact_E_Cv(8, t)[1] for t in Tf])
k_ex = np.argmax(C_ex)
print(f"\nC_v/N peak (L = 8): WL T = {Tf[k_wl]:.3f} (height {thermodynamic(E8, lg8, [Tf[k_wl]])['Cv'][0] / 64:.3f}), "
      f"exact T = {Tf[k_ex]:.3f} (height {C_ex[k_ex]:.3f}); T_c(infinite) = 2.269")
rel = lambda q, j: np.max(np.abs(np.array(wl[q]) / 64 / ex[:, j] - 1))
print(f"largest relative difference WL vs exact on this grid: <E> {100 * rel('E_mean', 0):.1f} %, "
      f"C_v {100 * rel('Cv', 1):.1f} %")

## `exercises/E4_5c_wang_landau_1t.py` — โค้ดที่ 4.7

E4.5 (extra)  The 1/t variant of Belardinelli and Pereyra (2007) against the original

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E4_5c_wang_landau_1t.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E4_5c_wang_landau_1t.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E4.5 (extra)  The 1/t variant of Belardinelli and Pereyra (2007) against the original
halving schedule of code:wl, on the 4 x 4 lattice where g(E) is known exactly.

The 1/t variant runs code:wl unchanged until ln f falls below 1/t, where
t = (spin-flip attempts) / (number of energy levels).  From then on ln f = 1/t at every
step and the flatness check is no longer used.  The run stops when ln f < ln_f_final.

    python exercises/E4_5c_wang_landau_1t.py          # seeds 1-8, about 3 minutes
    python exercises/E4_5c_wang_landau_1t.py --long   # adds 1/t down to 1e-6, about 20 minutes
"""
import os
import sys
import time
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
src = (HERE / '08_wang_landau_ising.py').read_text()
exec(src[src.index('rng = np.random.default_rng(seed=2026)'):src.index('E_bins, log_g, n_sweeps = wang_landau_2d_ising')])


def wang_landau_1t(L, ln_f_final=1e-6, flatness=0.8, check_every=1000):
    lattice = rng.choice([-1, 1], size=(L, L))
    N = L * L
    E_min = -2 * N
    E_bins = np.arange(E_min, 2 * N + 1, 4)
    log_g = np.zeros(len(E_bins))
    H = np.zeros(len(E_bins), dtype=int)
    visited = np.zeros(len(E_bins), dtype=bool)
    ln_f = 1.0
    idx = lambda E: (E - E_min) // 4
    E_curr = total_energy(lattice)
    sweep, attempts, one_over_t = 0, 0, False
    while ln_f > ln_f_final:
        for _ in range(N):
            i, j = rng.integers(0, L, size=2)
            s = lattice[i, j]
            nb = (lattice[(i+1) % L, j] + lattice[(i-1) % L, j]
                  + lattice[i, (j+1) % L] + lattice[i, (j-1) % L])
            E_new = E_curr + 2 * s * nb
            if rng.random() < np.exp(log_g[idx(E_curr)] - log_g[idx(E_new)]):
                lattice[i, j] = -s
                E_curr = E_new
            k = idx(E_curr)
            log_g[k] += ln_f
            H[k] += 1
            visited[k] = True
            attempts += 1
            if one_over_t:
                ln_f = visited.sum() / attempts          # ln f = 1/t
        sweep += 1
        if not one_over_t and sweep % check_every == 0:
            h = H[visited]
            if h.min() >= flatness * h.mean():
                ln_f /= 2.0
                H[:] = 0
                if ln_f < visited.sum() / attempts:      # halving has gone below 1/t
                    one_over_t = True
    E_bins, log_g = E_bins[visited], log_g[visited]
    log_g += np.log(2.0) - log_g[0]
    return E_bins, log_g, sweep


runs = [('original, ln f = 1e-6', wang_landau_2d_ising, 1e-6),
        ('1/t,      ln f = 1e-5', wang_landau_1t, 1e-5)]
if '--long' in sys.argv:
    runs.append(('1/t,      ln f = 1e-6', wang_landau_1t, 1e-6))

summary = {}
for name, fn, lnf in runs:
    print(f"\n{name}")
    print("seed   sweeps   time    sum g vs 2^16")
    errs, sweeps, secs = [], [], []
    for seed in range(1, 9):
        rng = np.random.default_rng(seed)
        t0 = time.time()
        E, lg, n = fn(4, ln_f_final=lnf)
        dt = time.time() - t0
        err = 100 * (np.exp(lg).sum() / 2**16 - 1)
        errs.append(err); sweeps.append(n); secs.append(dt)
        print(f"{seed:4d} {n:8d} {dt:6.1f} s   {err:+6.1f} %")
    summary[name] = (np.max(np.abs(errs)), np.mean(sweeps), np.mean(secs))
    print(f"largest |error| {summary[name][0]:.1f} %, mean {summary[name][1]:.0f} sweeps")

base = summary[runs[0][0]][1]
print("\nsummary (seeds 1-8)")
for name in summary:
    e, n, _ = summary[name]
    print(f"  {name}: largest |error| {e:4.1f} %, {n / base:4.1f} x the sweeps of the original")
print("-> 1/t keeps lowering the error as the run gets longer; the original schedule saturates")

## `exercises/E4_7_eda_correlations.py`

E4.7  The two most correlated pairs of features, with 95 % CIs by Fisher's z and by

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E4_7_eda_correlations.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E4_7_eda_correlations.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E4.7  The two most correlated pairs of features, with 95 % CIs by Fisher's z and by
bootstrap, and a check for a third variable behind the correlation.
"""
import os
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent.parent)
df = pd.read_csv('data/processed/galaxy10_features.csv')
num = df[['redshift', 'r_mag', 'g_mag', 'concentration']]
pairs = sorted(combinations(num.columns, 2), key=lambda p: -abs(num[list(p)].dropna().corr().iloc[0, 1]))
rng = np.random.default_rng(47)
for a, b in pairs[:2]:
    d = num[[a, b]].dropna()
    r, n = d.corr().iloc[0, 1], len(d)
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
    boot = [d.iloc[rng.integers(0, n, n)].corr().iloc[0, 1] for _ in range(1000)]
    blo, bhi = np.percentile(boot, [2.5, 97.5])
    print(f"{a} vs {b}: r = {r:.3f} (n = {n}); Fisher 95 % CI [{lo:.3f}, {hi:.3f}]; "
          f"bootstrap [{blo:.3f}, {bhi:.3f}]")
d = df.dropna(subset=['redshift', 'r_mag', 'g_mag'])
res_r = d['r_mag'] - np.polyval(np.polyfit(d['redshift'], d['r_mag'], 1), d['redshift'])
res_g = d['g_mag'] - np.polyval(np.polyfit(d['redshift'], d['g_mag'], 1), d['redshift'])
print(f"\nr_mag vs g_mag after removing the common dependence on redshift: "
      f"r = {np.corrcoef(res_r, res_g)[0, 1]:.3f}")
print("-> distance (redshift) drives both magnitudes; the remaining correlation comes from"
      " each galaxy's own luminosity and colour. Missing redshifts are not random (Ch.3), so"
      " the redshift correlation describes only the brighter part of the sample.")